In [ ]:
from mytools import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import re

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, MinMaxScaler, FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer

print("Imported all libraries successfully!")

In [ ]:
traindf = load_csv("train")
moviesdf = load_csv("movies")
traindf.shape, moviesdf.shape

## Examine movies.csv data  

In [ ]:
moviesdf.head()

In [ ]:
moviesdf.columns

In [ ]:
moviesdf.info()

In [ ]:
moviesdf.isnull().sum()

In [ ]:
moviesdf.describe()

In [ ]:
moviesdf["genre"].value_counts()

## Drop duplicates from moviesdf dataframe  

In [ ]:
movies_unique = moviesdf.drop_duplicates(subset=["movieid"])
movies_unique.shape, moviesdf.shape

## Merge traindf and moviesdf  

In [ ]:
train_movies_merged = pd.merge(traindf, movies_unique, on="movieid")
train_movies_merged.shape

In [ ]:
train_movies_merged.columns

In [ ]:
train_movies_merged = train_movies_merged[['movieid', 'reviewerName', 'isFrequentReviewer', 'reviewText',
       'title', 'audienceScore', 'rating', 'ratingContents',
       'releaseDateTheaters', 'releaseDateStreaming', 'runtimeMinutes',
       'genre', 'originalLanguage', 'director', 'boxOffice', 'distributor',
       'soundType', 'sentiment']]
train_movies_merged.shape

In [ ]:
train_movies_merged.head()

In [ ]:
train_movies_merged.isnull().sum()

## Clean data in merged df  

In [ ]:
# Fill missing values in "reviewText", 'rating" column with empty string and "NA" respectively
# Clean language names

train_final = train_movies_merged.copy()
train_final["reviewText"] = train_final["reviewText"].fillna(" ")
train_final["rating"] = train_final["rating"].fillna("NA")
train_final["originalLanguage"].replace({"English (United Kingdom)": "English", 
                                         "English (Australia)" : "English",
                                         "French (France)": "French", 
                                         "French (Canada)": "French",
                                         "Portuguese (Brazil)": "Portuguese",
                                         "Spanish (Spain)": "Spanish"},                                         
                                         inplace=True)
train_final["reviewText"].isna().sum()

In [ ]:
train_final["rating"].value_counts()

In [ ]:
train_final["genre"].value_counts()

In [ ]:
train_final["originalLanguage"].unique(), train_final["originalLanguage"].value_counts()

In [ ]:
train_final.columns

## Keep only the columns to work on  

In [ ]:
train_final = train_final.drop(columns=["title", "ratingContents", "releaseDateTheaters", "releaseDateStreaming", "boxOffice", "distributor", "soundType"])
train_final.shape,  train_final.columns

## Separate features and labels  

In [ ]:
train_features = train_final.iloc[:, :-1]
train_labels = train_final.iloc[:, -1]
train_features.shape, train_labels.shape

In [ ]:
train_features.head()

In [ ]:
train_labels.head()

## Try "select_features" function from mytools module  

In [ ]:
df = select_features(load_csv("train"), load_csv("movies"))
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
dftest = select_features(load_csv("test"), load_csv("movies"))
dftest.head()

In [ ]:
dftest.shape

In [ ]:
dftest.columns

In [ ]:
df.columns

## Train models  

In [ ]:
train1_fs = df[['movieid', 
                'reviewerName', 
                'isFrequentReviewer', 
                'reviewText',
                'audienceScore', 
                'rating', 
                'runtimeMinutes', 
                'genre', 
                'originalLanguage', 
                'director'
                ]]

# train1_fs = df[['reviewText',
#                 'audienceScore', 
#                 'runtimeMinutes', 
#                 'genre', 
#                 'originalLanguage', 
#                 'director'
#               ]]

train1_labels = df['sentiment']
train1_fs.shape, train1_labels.shape

In [ ]:
# train1_fs.loc[:, "audienceScore"] = SimpleImputer(strategy='mean', missing_values=np.nan).fit_transform(train1_fs[["audienceScore"]])
# train1_fs.loc[:, "runtimeMinutes"] = SimpleImputer(strategy='mean', missing_values=np.nan).fit_transform(train1_fs[["runtimeMinutes"]])
# train1_fs.loc[:, "genre"] = SimpleImputer(strategy='most_frequent').fit_transform(train1_fs[["genre"]])
# train1_fs.loc[:, "originalLanguage"] = SimpleImputer(strategy='constant', fill_value='Unknown').fit_transform(train1_fs[["originalLanguage"]])
# train1_fs.isnull().sum()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train1_fs, train1_labels, test_size=0.25, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
train1_fs.info()

In [ ]:
train1_fs.isnull().sum()

In [ ]:
# ct1_imputer = ColumnTransformer(transformers=[
#                                 ('imputer_audienceScore', SimpleImputer(strategy='mean', missing_values=np.nan), ['audienceScore']),
#                                 ('imputer_runtimeMinutes', SimpleImputer(strategy='mean', missing_values=np.nan), ['runtimeMinutes']),
#                                 ('imputer_genre', SimpleImputer(strategy='most_frequent'), ['genre']),
#                                 ('imputer_lang', SimpleImputer(strategy='constant', fill_value='Unknown'), ['originalLanguage']),
#                                 ], 
#                                 remainder='passthrough')

In [ ]:
pipe_imputer_num = Pipeline(steps=[
                                ('imp1', SimpleImputer(strategy='mean', missing_values=np.nan)),
                                # ('toarray', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True))
                            ])

pipe_imputer_freq = Pipeline(steps=[
                                ('imp2', SimpleImputer(strategy='most_frequent')),
                                ('toarray', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True))
                            ])

pipe_imputer_const = Pipeline(steps=[
                                ('imp3', SimpleImputer(strategy='constant', fill_value='Unknown')),
                                ('toarray', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True))
                            ])

# pipe_tvec = Pipeline(steps=[
#                             ('tvec', TfidfVectorizer(max_features=10000)),
#                             ('toarray', FunctionTransformer(lambda x: x.toarray(), accept_sparse=True))
#                             ])

vectorizer_params = dict(max_features=None, ngram_range=(1, 2), stop_words='english')
pipe_text = Pipeline(steps=[
                            # ('imp', SimpleImputer(strategy='most_frequent')),
                            # ("squeez", FunctionTransformer(lambda x: x.squeeze())),
                            # ("vect", CountVectorizer(**vectorizer_params)),
                            # ("tfidf", TfidfTransformer()),
                            ("tvec", TfidfVectorizer(**vectorizer_params)),
                            # ("toarray", FunctionTransformer(lambda x: x.toarray())),
                        ])

In [ ]:
train1_fs.isna().sum()

In [ ]:
train1_fs.columns

In [ ]:
ct2 = ColumnTransformer(transformers=[
                                    ('num', pipe_imputer_num, ['audienceScore', 'runtimeMinutes']),
                                    # ('text', pipe_text, 'director'),
                                    ('reviewText', pipe_text, 'reviewText'),
                                    ], 
                                    remainder='drop', sparse_threshold=0.3)

In [ ]:
# ct1 = ColumnTransformer(transformers=[
#                         ('tvec_movieid', TfidfVectorizer(), ['movieid']),
#                          ('tvec_reviewerName', TfidfVectorizer(max_features=10000), ['reviewerName']),
#                         #  ('ohe_freqRev', OneHotEncoder(handle_unknown='ignore'), ['isFrequentReviewer']),
#                         ('tvec_isFreq', TfidfVectorizer(), ['isFrequentReviewer']),
#                          ('tvec_reviewText', TfidfVectorizer(ngram_range=(1,2), max_features=10000), ['reviewText']),
#                         #  ('std_scaler_audienceScore', StandardScaler(), ['audienceScore']),
#                         #  ('ohe_rating', OneHotEncoder(handle_unknown='ignore'), ['rating']),
#                         ('tvec_rating', TfidfVectorizer(), ['rating']),
#                         #  ('mm_scaler_runtimeMinutes', MinMaxScaler(), ['runtimeMinutes']),
#                          ('tvec_genre', TfidfVectorizer(max_features=20), ['genre']),
#                          ('tvec_originalLanguage', TfidfVectorizer(max_features=100), ['originalLanguage']),
#                          ('tvec_director', TfidfVectorizer(max_features=10000), ['director']),
#                          ], remainder='passthrough', sparse_threshold=0.3)
# ct1

In [ ]:
pipe1 = Pipeline(steps=[
                        # ('imputer', ct1_imputer),
                        ('transformer', ct2), 
                        ('logreg', LogisticRegression(C=2, max_iter=1000))
                        ])

In [ ]:
pipe1.fit(X_train, y_train)

In [ ]:
pipe1.score(X_train, y_train)

In [ ]:
predict_n_evaluate(pipe1, X_test, y_test)

### test file

In [ ]:
# test1 = dftest[['reviewerName', 'isTopCritic', 'reviewText']]
# test1.shape

In [ ]:
X_train.columns

In [ ]:
X_train["rating"].value_counts()

In [ ]:
vec = TfidfVectorizer()
yyyy = vec.fit_transform(X_train["originalLanguage"].astype(str))
ypd = pd.DataFrame(yyyy.toarray())
ypd

In [ ]:
ypd.sum()

In [ ]:
vec.get_feature_names()

In [ ]:
vec.vocabulary_

In [ ]:
vec.get_params()